# Synthetic results: controls, interactions and omega

**Current entry point.** The three primary RAPID responses are **peak lag, plateau-to-plateau event duration, and peak excess**, each expressed as child/parent percentage change. Definitions are fixed in [the shared metric contract](../../RAPID/CURRENT_RESPONSE_METRICS.md). Duration uses a 1% parent-amplitude tolerance shared by each pair, with 0.5% and 2% checks. Temporal spread is retained as a diagnostic.

This notebook answers **what controls these responses** and **why omega leaves scatter**. Prediction is a separate result in [synthetic_predictor_validation.ipynb](synthetic_predictor_validation.ipynb). The [roadmap](../ANALYSIS_ROADMAP.md) identifies what remains before a final real-network validation.

The full balanced grid contains **15,309 comparisons: 2,187 configurations × seven collapse treatments**. This notebook reads the completed selected hydraulic rerun. A separate small numerical audit is explicitly identified below.

The emerging story is:

1. Allocation matters across all three responses; extent and slope matter more for shape than their omega invariance suggests.
2. Omega discards absolute scale: width, slope and extent can vary while omega stays fixed. This explains why the discrete grid creates vertical bands, but does not by itself establish the physical cause of each response.
3. Some scatter is numerical: a tested short network activates the length floor. Other responses depend on the forcing timescale. Both should be separated from geometry before making universal claims.

Section 5 integrates the width/length and fixed-unit embedding extension. Its completed full-run status is reported independently of the original balanced grid; predictor transfer is evaluated in the existing predictor notebook.

**Checkpoint · 29 September 2026:** [SYN-2026-09-29](../notes/2026-09-29_synthetic_checkpoint.md) records the findings, the pause pending real-case comparison, and the restart of hydraulic applicability.

**Current synthetic version · 30 September 2026:** BASED reference depth, local width–depth exponent **α=0.65**, Manning n=0.035, X=0.1 and 200m segmentation/floor. Responses and local-kb resistance are recomputed from the versioned routing run. Geometry and validation partitions are unchanged. **Real-case comparison is deferred until the updated-slope runs.** Earlier outputs are preserved separately; the old-versus-new comparison changes the complete hydraulic setup, not alpha alone.


**Population correction — ALL-PAIRS-LAG-2026-10-05:** real timing prediction and synthetic-to-real transfer now include every one of the **2,214 distinct parent–child pairs**, with no edge-compound or path-count exclusions. The previous 1,652-pair analysis was filtered. Synthetic factorial sensitivity below is unchanged. See [predictor validation](synthetic_predictor_validation.ipynb) and the [current report](../../RAPID/validation/all_pairs_lag_20261005/README.md).

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm,TwoSlopeNorm,Normalize
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'synthetic_experiments').exists())
import sys
sys.path.insert(0,str(ROOT/'synthetic_experiments/investigations'))
from analysis_paths import results_root
ACTIVE_RUN=results_root()/'full_grid_island'
ACTIVE_EXTENSION=results_root()/'embedded_width_length_extension'
assert json.loads((ACTIVE_RUN/'manifest.json').read_text())['config']['selected_hydraulic_reference']['depth_alpha']==.65
OUT=ACTIVE_RUN/'current_analysis'
d=pd.read_parquet(OUT/'current_comparisons.parquet')
Y=['lag_shift_pct','event_duration_change_pct','peak_excess_change_pct']
YL=['Peak-lag change','Event-duration change','Peak-excess change']
FACTORS=['total_width_m','reference_slope','outer_a_fraction','inner_fraction','length_multiplier','total_length_m','internal_location']
FL=['Total width','Reference slope','Outer A fraction','Inner / crossover fraction','Branch length multiplier','Network extent','Internal location']
F=dict(zip(FACTORS,FL))
STATES=['simple/full','nested/inner_only','nested/main_island_only','nested/full','crossover/upstream_only','crossover/downstream_only','crossover/full']
SL=['Simple: full','Nested: inner','Nested: main island','Nested: full','Cross: upstream','Cross: downstream','Cross: full']
SN=dict(zip(STATES,SL))
plt.rcParams.update({'axes.spines.top':False,'axes.spines.right':False,'figure.dpi':110})
def heat(ax,a,rows,cols,title,vmin,vmax,cmap='YlOrRd',fmt='.1f'):
    a=np.asarray(a,float);norm=TwoSlopeNorm(vmin=vmin,vcenter=0,vmax=vmax) if vmin<0<vmax else Normalize(vmin,vmax)
    im=ax.imshow(a,aspect='auto',norm=norm,cmap=cmap)
    ax.set_xticks(range(len(cols)),cols,rotation=40,ha='right',fontsize=8)
    ax.set_yticks(range(len(rows)),rows,fontsize=9);ax.set_title(title,loc='left',fontsize=11)
    for i,j in np.ndindex(a.shape):
        if np.isfinite(a[i,j]):
            color='white' if np.dot(im.cmap(im.norm(a[i,j]))[:3],[.2126,.7152,.0722])<.48 else '#222'
            ax.text(j,i,format(a[i,j],fmt),ha='center',va='center',fontsize=7,color=color)
    return im
def savefig(fig,name):
    fig.savefig(OUT/name,dpi=170,bbox_inches='tight');display(fig);plt.close(fig)

print(f'{len(d):,} comparisons; three current responses; 21 unique factor pairs.')
display(d.groupby('child_state')[Y].agg(['min','median','max']).round(2))
a=d[['duration_change_005_pct','duration_change_010_pct','duration_change_020_pct']]
print('Duration threshold range (median / max pp):', (a.max(axis=1)-a.min(axis=1)).median(),(a.max(axis=1)-a.min(axis=1)).max())
flips=np.sign(a).nunique(axis=1)>1
print('Cases whose duration direction changes with tolerance:',int(flips.sum()),'of',len(d))
display(d.loc[flips].groupby('child_state').event_duration_change_pct.agg(['count','min','max']).round(3))


## 1. Decomposed variation and direction

Main effects average over all other factors. Each pair effect subtracts both main effects; three-or-more-factor terms form the remainder. Main + distinct pairs + remainder = 100% within each response and treatment.

**Variance contributions cannot be negative.** Total sensitivity includes every interaction involving that factor and overlaps other totals, so total sensitivities must not be summed. The separate signed panel reports matched high-minus-low response changes, holding the other six factors equal. Its −100 to +100 percentage-point colour scale does not clip the underlying response values.

Read the current duration-threshold summary above: small effects can change sign with tolerance. The width/length mechanism is extended to more unequal branches in section 5.3. The factorial calculation remains restricted to the complete original grid; the separately designed embedding cases are integrated into the interpretation rather than pooled into an unbalanced variance calculation.

In [ ]:
main=pd.read_csv(OUT/'main_total_effects.csv')
pairs=pd.read_csv(OUT/'pair_interactions.csv')
partition=pd.read_csv(OUT/'variance_partition.csv')
fig,axes=plt.subplots(2,3,figsize=(19,10),constrained_layout=True)
for j,(response,label) in enumerate(zip(Y,YL)):
    for i,(column,title) in enumerate([('main_share','Main contribution'),('total_share','Total sensitivity')]):
        matrix=main[main.response==response].pivot(index='factor',columns='child_state',values=column).reindex(index=FACTORS,columns=STATES)*100
        im=heat(axes[i,j],matrix,FL if j==0 else ['']*7,SL,label+' · '+title,0,100)
fig.colorbar(im,ax=axes,label='Response variance (%)',shrink=.65)
savefig(fig,'current_factor_variance.png')
display(partition.assign(main_pct=100*partition.main_share,pair_pct=100*partition.pair_share,higher_pct=100*partition.higher_share)[['child_state','response','main_pct','pair_pct','higher_pct']].round(1))
contrasts=pd.read_csv(OUT/'matched_contrasts.csv')
fig,axes=plt.subplots(1,3,figsize=(19,5.2),constrained_layout=True)
limit=max(100,float(contrasts['mean'].abs().max()))
for j,(ax,response,label) in enumerate(zip(axes,Y,YL)):
    matrix=contrasts[contrasts.response==response].pivot(index='factor',columns='child_state',values='mean').reindex(index=FACTORS,columns=STATES)
    im=heat(ax,matrix,FL if j==0 else ['']*7,SL,label,-limit,limit,'RdBu_r',fmt='+.1f')
fig.colorbar(im,ax=axes,label='Mean matched high-minus-low change (percentage points)',shrink=.7)
savefig(fig,'current_signed_effects.png')


## 2. All 21 pair interactions

These are additional variance contributions after subtracting both main effects. Main effects plus all distinct pairs plus the higher-order remainder sum to 100%. The panels below are recalculated for the selected hydraulic setup; no legacy ranking is assumed.

In [ ]:
from itertools import combinations
pair_order=list(combinations(FACTORS,2));pair_names=[F[a]+' × '+F[b] for a,b in pair_order]
pairs['pair']=[F[a]+' × '+F[b] for a,b in zip(pairs.factor_a,pairs.factor_b)]
fig,axes=plt.subplots(1,3,figsize=(20,11),constrained_layout=True)
upper=float(pairs.pair_share.max()*100)
for j,(ax,response,label) in enumerate(zip(axes,Y,YL)):
    matrix=pairs[pairs.response==response].pivot(index='pair',columns='child_state',values='pair_share').reindex(index=pair_names,columns=STATES)*100
    im=heat(ax,matrix,pair_names if j==0 else ['']*21,SL,label,0,upper)
fig.colorbar(im,ax=axes,label='Pair contribution to response variance (%)',shrink=.65)
savefig(fig,'current_21_interactions.png')
display(pairs.sort_values('pair_share',ascending=False).groupby('response').head(3).assign(variance_pct=lambda x:100*x.pair_share)[['response','child_state','pair','variance_pct']].round(2))


## 3. Omega and the vertical bands

$\omega=L_f E[-\log_{10}(\Lambda_K)]$, with resistance recomputed using local kb. The plots and matched-control tables test which factors leave omega unchanged while altering routed lag. Repeated values and residual spread are features of this controlled model experiment, not by themselves evidence for a universal physical law.

Section 5.2 varies surrounding length while holding the unit fixed. Section 6 tests whether the timing response itself should retain that network-scale dependence.

In [ ]:
fig,axes=plt.subplots(3,3,figsize=(14,10),constrained_layout=True)
for i,state in enumerate(['simple/full','nested/inner_only','crossover/full']):
    g=d[d.child_state==state]
    for j,f in enumerate(['total_width_m','reference_slope','total_length_m']):
        ax=axes[i,j]
        im=ax.scatter(g.omega,g.lag_shift_pct,c=g[f],norm=LogNorm(g[f].min(),g[f].max()),cmap='viridis',s=8,alpha=.55,rasterized=True)
        ax.set(title=SN[state]+' · '+F[f],xlabel='Omega',ylabel='Peak-lag change (%)' if j==0 else '')
        ax.grid(alpha=.12)
        fig.colorbar(im,ax=ax,shrink=.7)
savefig(fig,'omega_colored_controls.png')
bands=pd.read_csv(OUT/'omega_bands.csv')
display(bands[['child_state','scope','n_bands','within_variance_fraction','max_lag_range_pp']].round(3))
matched=pd.read_csv(OUT/'omega_matched_controls.csv')
display(matched[matched.scope=='all'].groupby('factor')[['n_complete_contexts','n_unchanged_omega_contexts','median_lag_range_pp','p90_lag_range_pp']].mean().round(3))


## 4. Separate geometry, numerics and forcing

The matched nested/full audit reruns seven geometries under the selected depth law. Refinement changes both segment length and its effective-length floor; X and forcing-duration changes are separate model/forcing sensitivities. Read the current tables for changes and flags rather than carrying forward the old hydraulic setup’s numerical conclusions.

In [ ]:
audit=pd.read_csv(OUT/'omega_numerical_check.csv')
order=audit.case.drop_duplicates().tolist()
case_labels=['Baseline','Width 250','Width 1000','Slope 5e−5','Slope 5e−4','Extent 10 km','Extent 40 km']
fig,axes=plt.subplots(1,3,figsize=(16,5),constrained_layout=True)
styles={'baseline_200m':('200 m','#334155','-'),'refined_100m':('100 m','#60a5fa','-'),'refined_50m':('50 m','#2563eb','-'),'double_pulse_duration':('Double pulse duration','#ea580c','--')}
for ax,y,label in zip(axes,Y,YL):
    for variant,(name,color,style) in styles.items():
        g=audit[audit.variant==variant].set_index('case').reindex(order)
        ax.plot(range(len(order)),g[y],marker='o',color=color,ls=style,label=name)
    ax.set(xticks=range(len(order)),xticklabels=case_labels,title=label,ylabel='Child/parent change (%)')
    ax.tick_params(axis='x',rotation=45);ax.grid(alpha=.15)
axes[0].legend(fontsize=8)
savefig(fig,'omega_numerical_audit.png')
base=audit[audit.variant=='baseline_200m'].set_index('case')[Y]
changes=[]
for variant,g in audit.groupby('variant'):
    row=(g.set_index('case')[Y]-base).abs().max().to_dict();row['variant']=variant;changes.append(row)
display(pd.DataFrame(changes).set_index('variant').round(3))
display(d.groupby('child_state')[['any_cap','any_floor','within_one_dt']].sum())


## 5. Extension: fixed-unit embedding, unequal widths and longer branches

This separate extension tests whether more unequal width splits and a smaller affected fraction bring synthetic comparisons into the low-omega, small-lag-change range occupied by real cases.

**Design:** outer A fractions 0.05, 0.10, 0.20, 0.50, 0.80, 0.90, 0.95; selected branch length multipliers 1, 1.25, 1.5, 2, 3; total extents 20, 100, 400, 800 km. There are 140 configurations and 980 planned comparisons across the seven existing treatments. The outer unit retains its 10 km span; only serial approach and exit lengths increase. Width=500 m, slope=1e−4, inner fraction=0.5 and internal junction positions remain fixed.

Both width orientations matter: in a simple unit, A=0.95 lengthens the dominant branch, whereas A=0.05 lengthens the narrow branch. Nested and crossover geometries retain their existing family-specific length targets.

All 140 configurations / 980 comparisons are now routed under the selected settings. The extension stays separate from the balanced factorial decomposition. See [the design](../EMBEDDED_EXTENSION.md).

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'synthetic_experiments').exists())
EXTENSION_OUT=ACTIVE_EXTENSION
extension_manifest=json.loads((EXTENSION_OUT/'manifest.json').read_text())
extension_plan=pd.read_csv(EXTENSION_OUT/'plan.csv')
keys=['configuration_id','child_state','parent_run_id','child_run_id']
extension_data=None
for phase in ['full','pilot']:
    file=EXTENSION_OUT/f'{phase}_results.csv'
    if not file.exists():continue
    candidate=pd.read_csv(file)
    expected=extension_plan if phase=='full' else extension_plan[extension_plan.is_pilot]
    if len(candidate)==len(expected) and len(candidate[keys].merge(expected[keys],on=keys,validate='one_to_one'))==len(expected):
        extension_data=candidate;EXTENSION_PHASE=phase;break
if extension_data is None:raise RuntimeError('Run the matching pilot or full phase before opening results.')
EXTENSION_Y=['lag_shift_pct','event_duration_change_pct','peak_excess_change_pct']
print(f'SHOWING {EXTENSION_PHASE.upper()}: {extension_data.configuration_id.nunique()} configurations, {len(extension_data)} comparisons.')
print(f'Full plan: {extension_plan.configuration_id.nunique()} configurations, {len(extension_plan)} comparisons.')
plt.rcParams.update({'axes.spines.top':False,'axes.spines.right':False,'axes.grid':True,'grid.alpha':.15})
display(extension_data.groupby('child_state')[['omega','length_fraction',*EXTENSION_Y]].agg(['min','median','max']).round(4))

extension_plan=extension_data.copy()  # complete routed plan, including recomputed resistance
extension_data['max_volume_error']=extension_data[['parent_volume_error','child_volume_error']].abs().max(axis=1)
extension_data['parent_floored_segments']=extension_data.parent_n_length_floored
extension_data['child_floored_segments']=extension_data.child_n_length_floored


### 5.1. Network geometry and omega

Across embeddings of the same unit, local resistance ratio and exposure stay fixed. The change in omega is therefore exactly the change in domain/network channel-length fraction. Total head drop increases with extent so that the unit's reference slope and head loss remain fixed.

The table below uses the complete geometric plan; it requires no routing results. It illustrates the new range for equal-length simple branches. Network extent is not the denominator of length fraction: that denominator sums channel lengths, including parallel branches.

In [ ]:
simple=extension_plan[(extension_plan.child_state=='simple/full')&(extension_plan.length_multiplier==1)&extension_plan.outer_a_fraction.isin([.05,.5,.95])]
display(simple[['outer_a_fraction','network_extent_m','approach_length_m','exit_length_m','length_fraction','resistance_predictor']].sort_values(['outer_a_fraction','network_extent_m']).round(5))
# Verify the key controlled comparison using every planned geometry.
for _,g in extension_plan.groupby(['child_state','outer_a_fraction','length_multiplier']):
    g=g.sort_values('network_extent_m')
    assert np.allclose(g.Lambda_matched,g.Lambda_matched.iloc[0])
    assert np.allclose(g.width_fraction,g.width_fraction.iloc[0])
    assert (g.length_fraction.diff().dropna()<0).all()
print('Local resistance/exposure invariance and declining length fraction verified for all planned embeddings.')


### 5.2. Synthetic omega–lag range

Both the main grid and extension use the selected hydraulic setup. The earlier real pilot is deliberately omitted: its resistance/depth assumptions and slopes have not been updated. Real-case comparison waits for the new slope-based runs.

In [ ]:
extension_original=pd.read_parquet(ACTIVE_RUN/'current_analysis/current_comparisons.parquet')
fig,axes=plt.subplots(1,2,figsize=(13,5),constrained_layout=True)
colors=plt.get_cmap('viridis')(np.linspace(.15,.85,extension_data.network_extent_m.nunique()))
for ax in axes:
    ax.scatter(extension_original.omega,extension_original.lag_shift_pct,c='#cbd5e1',s=5,alpha=.25,label='Selected full grid',rasterized=True)
    for color,(extent,g) in zip(colors,extension_data.groupby('network_extent_m')):
        ax.scatter(g.omega,g.lag_shift_pct,color=color,s=28,label=f'{EXTENSION_PHASE}: {extent/1000:g} km',zorder=3)
    ax.set(xlabel='Omega',ylabel='Peak-lag change (%)')
axes[0].set_title('Entire range');axes[1].set(xlim=(-.005,.16),ylim=(-12,2),title='Low-omega / small-change region')
axes[0].legend(fontsize=8)
fig.savefig(EXTENSION_OUT/f'{EXTENSION_PHASE}_omega_lag.png',dpi=170);display(fig);plt.close(fig)
summary=[]
for label,g,x in [('Original grid',extension_original,'omega'),('Extension '+EXTENSION_PHASE,extension_data,'omega')]:
    summary.append(dict(dataset=label,omega_min=g[x].min(),omega_median=g[x].median(),omega_max=g[x].max(),median_abs_lag=g.lag_shift_pct.abs().median(),fraction_omega_below_01=(g[x]<.1).mean(),length_fraction_min=g.length_fraction.min(),length_fraction_max=g.length_fraction.max()))
display(pd.DataFrame(summary).round(4))

### 5.3. Dominant versus narrow lengthened branch

These panels retain simple/full to make branch orientation unambiguous. Extent varies with the same local geometry; multiplier changes are separate local interventions. Lines connect the completed extension levels.

The three responses are the current lag, 1% plateau duration and peak excess. Parent and child share the same absolute plateau tolerances based on the parent peak amplitude.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(15,4.5),constrained_layout=True)
for a,color in [(.05,'#2563eb'),(.95,'#ea580c')]:
    for multiplier,style in [(1,'-'),(3,'--')]:
        g=extension_data[(extension_data.child_state=='simple/full')&(extension_data.outer_a_fraction==a)&(extension_data.length_multiplier==multiplier)].sort_values('network_extent_m')
        for ax,y,label in zip(axes,EXTENSION_Y,['Peak-lag change','Event-duration change','Peak-excess change']):
            ax.plot(g.network_extent_m/1000,g[y],marker='o',ls=style,color=color,label=f'A/B {a*100:g}/{(1-a)*100:g}, A ×{multiplier}')
            ax.set(xscale='log',xlabel='Network extent (km)',ylabel='Child/parent change (%)',title=label)
axes[0].legend(fontsize=8)
fig.suptitle('Fixed 10 km simple unit; grow the surrounding network')
fig.savefig(EXTENSION_OUT/f'{EXTENSION_PHASE}_embedding_responses.png',dpi=170);display(fig);plt.close(fig)


### 5.4. Resolution and hydraulic safeguards

The selected extension activates neither celerity caps nor effective-length floors. Peak-time differences of one timestep or less remain flagged; a small value is not a precisely resolved response. All cases pass volume balance, spin-up and final-recession checks. The duration-threshold envelope below remains relevant.

Section 6 uses these same saved networks to compare timing definitions. Extension observations do not enter the original factorial decomposition or the fixed-model training.

In [ ]:
qa=extension_data.assign(any_cap=(extension_data.parent_capped_segments+extension_data.child_capped_segments)>0,any_floor=(extension_data.parent_floored_segments+extension_data.child_floored_segments)>0)
display(qa.groupby(['network_extent_m']).agg(comparisons=('configuration_id','size'),capped=('any_cap','sum'),floored=('any_floor','sum'),lag_within_one_dt=('within_one_dt','sum'),max_volume_error=('max_volume_error','max')))
v=extension_data[['duration_change_005_pct','duration_change_010_pct','duration_change_020_pct']]
print('Duration threshold range, median/max pp:',(v.max(axis=1)-v.min(axis=1)).median(),(v.max(axis=1)-v.min(axis=1)).max())
assert extension_data.max_volume_error.max()<1e-4

## 6. Which timing response should lead the results?

**SYN-2026-09-30-TIMING-OPTIONS.** Requested after the embedding experiment revealed a dependence on gauge/network extent. There is no uniquely best “lag change per metre”. The options answer different questions. This comparison introduces **no new unit length**: unit normalization uses the matched **parent equivalent length**, $L_{eq,p}$.

Write $r=T_c/T_p-1$, where $T_p,T_c$ are model inlet-to-outlet peak lags. $L_n$ is single-centerline network extent, not summed channel length. The four main comparison axes below are:

| Option | Definition | Interpretation |
|---|---|---|
| Percentage lag | $100r$ | Whole-network relative impact |
| Absolute timing | $(T_c-T_p)/3600$ | Model hours gained/lost; size-dependent control |
| Network speed change | $L_n(1/T_c-1/T_p)$ | Change in apparent average network speed, m/s |
| Timing / parent equivalent length | $(T_c-T_p)/L_{eq,p}$ | Network timing impact per metre of affected parent domain; displayed in min/km |

The parent length is held fixed for both states; it is width-weighted, not a newly defined width-free length. The last option does not measure travel time or celerity inside the unit itself. Negative timing changes mean earlier arrival, while positive speed changes mean faster propagation. The primary metric contract is **not yet replaced**; this is the basis for choosing a main-text response and supplementary alternatives.

In [ ]:
TIMING=OUT/'timing_response_comparison'
timing_provenance=json.loads((TIMING/'provenance.json').read_text())
timing_original=pd.read_parquet(TIMING/'original_metrics.parquet')
timing_extension=pd.read_parquet(TIMING/'extension_metrics.parquet')
timing_definitions=pd.read_csv(TIMING/'metric_definitions.csv').set_index('metric')
timing_core=['lag_change_pct','lag_change_hours','network_speed_change_mps','lag_per_parent_leq_min_km']
print(timing_provenance['algebra_checks'])
print('Original / extension comparisons:',len(timing_original),len(timing_extension))
print('Matched original-grid anchors verified:',timing_provenance['n_verified_anchors'])
fig,axes=plt.subplots(2,2,figsize=(13,8),constrained_layout=True)
examples=[('simple/full',.95,3.,'Simple: dominant long A'),('simple/full',.05,3.,'Simple: narrow long A'),('crossover/full',.8,1.5,'Compound: A=.8, multiplier1.5')]
for ax,metric in zip(axes.flat,timing_core):
 for state,a,m,label in examples:
  g=timing_extension[(timing_extension.child_state==state)&(timing_extension.outer_a_fraction==a)&(timing_extension.length_multiplier==m)].sort_values('network_extent_m')
  ax.plot(g.network_extent_m/1000,g[metric],'o-',label=label)
  weak=g.lag_shift_seconds.abs().le(3*g.timing_resolution_seconds)
  ax.scatter(g.loc[weak,'network_extent_m']/1000,g.loc[weak,metric],marker='x',color='black',s=55,zorder=5)
 ax.set(xscale='log',xlabel='Network extent (km)',ylabel=timing_definitions.loc[metric,'units'],title=timing_definitions.loc[metric,'label'])
 ax.set_xticks([20,100,400,800],['20','100','400','800'])
 ax.axhline(0,color='0.7',lw=.8);ax.grid(alpha=.15)
axes[0,0].legend(fontsize=8)
fig.suptitle('Keep the local geometry fixed; add approach and exit length · × = peak difference ≤3 timesteps')
fig.savefig(TIMING/'timing_options_embedding.png',dpi=170);display(fig);plt.close(fig)
example=timing_extension[(timing_extension.child_state=='simple/full')&(timing_extension.outer_a_fraction==.95)&(timing_extension.length_multiplier==3)]
display(example[['network_extent_m','parent_equivalent_length',*timing_core]].round(4))


### 6.1. What remains dependent on the surrounding network?

For each of the 245 fixed geometries, compare the four embeddings with its 20 km case. The summary reports maximum departure from that case divided by its absolute value. A value of 25% means the response varies by at most one quarter of its 20 km magnitude across the sampled embeddings; it is not the original lag percentage.

Relative drift is unstable near zero. The main summary therefore requires the underlying peak-time difference to exceed three timesteps at **all four extents**. Counts show the retained groups; all observations remain available in the exported tables and predictor analysis. This resolution screen is not an uncertainty interval.

Absolute timing and timing/parent-Leq necessarily have identical relative drift here because parent Leq is constant within an embedding group. Normalizing by Leq helps compare different local scales; it cannot correct changes in the peak position caused by downstream hydrograph reshaping.

These embeddings add approach and exit length together. They test surrounding-network sensitivity, rather than isolating a move of the downstream gauge alone.

In [ ]:
drift=pd.read_csv(TIMING/'embedding_summary.csv')
main_drift=drift[(drift.scope=='resolved_gt3dt')&drift.metric.isin(timing_core)].set_index('metric').reindex(timing_core).copy()
main_drift['median_departure_pct']=100*main_drift.median_relative_drift
main_drift['p90_departure_pct']=100*main_drift.p90_relative_drift
print('245 local geometries; each has four embeddings. All remain in the fitted-model evaluation.')
display(main_drift[['n_groups','n_nonzero_baselines','median_departure_pct','p90_departure_pct']].round(2))
centroid_check=pd.read_csv(TIMING/'centroid_embedding_diagnostic.csv')
display(centroid_check.groupby('timing').maximum_departure_seconds.agg(['median','max']))


The centroid check uses the already saved excess-volume-weighted event arrival times, **only as a diagnostic**. Their parent–child difference stays constant to numerical precision across these embeddings, while peak-time differences need not. This is consistent with common linear approach/exit routing adding the same mean delay while reshaping the peaks differently. It does not prove comparable invariance in nonlinear real rivers, and it does not silently replace peak lag with centroid timing.

### 6.2. Supplementary definitions and the reference-time assumption

No observed gauge timings exist for these synthetics. Two supplementary formulas use the saved **single-centerline control lag** $T_{ref}$ as a reference sensitivity: $c_{ref}=L_n/T_{ref}$ and $\Delta c_{ref}=c_{ref}[T_p/T_c-1]$, or $rT_{ref}/L_{eq,p}$. The centerline is not an observation of the parent network. Actual gauge-scaled performance remains untested until the real-case analysis.

For completeness, the proposed state-specific equivalent-speed index is $100[(L_{eq,c}/L_{eq,p})(T_p/T_c)-1]$. It changes both length and time, can remove the path-shortening effect, and combines local lengths with whole-network travel times. It is not a local celerity estimate. Its reciprocal slowness version is also exported.

The fixed-model comparison is in the predictor notebook. A higher R² is not sufficient to choose a response: normalization changes its meaning and introduces mathematical dependence on Leq, which is also a candidate predictor.

In [ ]:
display(timing_definitions[['label','units','formula','meaning']])
display(drift[drift.scope=='resolved_gt3dt'].set_index('metric')[['n_groups','median_relative_drift','p90_relative_drift']].round(4))


## Next decisions

Use the selected-run factorial analysis and fixed response definitions. The old-versus-new comparison below quantifies the complete hydraulic-setup change. Interpret model scores as prediction of deterministic routing outputs; retain the depth/allocation consistency limitation. Real-network validation and its hydraulic selection await new runs using updated slopes.

## Previous versus selected hydraulic setup

These changes include the move from kb=20/n=.35 to BASED-reference local kb/alpha=.65/n=.035. They are not an alpha-only contrast. Geometry, response definitions and validation partitions are matched; earlier outputs remain preserved.

In [ ]:
previous_selected=pd.read_csv(ACTIVE_RUN.parent/'old_new_response_summary.csv')
display(previous_selected.round(4))